# TP XGBoost — Des hyperparamètres au modèle final

On utilise `nycflights13::flights` (vols domestiques américains de 2013). Pour garder des temps d'exécution raisonnables, on travaille sur 50 000 vols.

**Objectif** : prédire si un vol arrive avec un retard de plus de 15 minutes.

Ce TP suit le fil : préparer les données → entraîner un premier modèle → comprendre ce que font les principaux hyperparamètres → laisser la validation croisée régler le modèle → évaluer sur le test → comparer à `gbm`.

In [ ]:
library(dplyr)
library(nycflights13)
library(xgboost)
library(pROC)
library(gbm)
library(knitr)

## 1. Prétraitement des données

Création de la variable cible `retard` qui vaut 1 si le retard à l'arrivée est strictement supérieur à 15 minutes et 0 sinon.

In [ ]:
data(flights)

set.seed(123)
flights_clean <- flights |>
  filter(!is.na(arr_delay), !is.na(dep_delay), !is.na(air_time)) |>
  mutate(retard = ifelse(arr_delay > 15, 1, 0),
         carrier = factor(carrier),
         origin = factor(origin)) |>
  select(retard, dep_delay, air_time, distance, month, day, hour, minute, carrier, origin) |>
  sample_n(50000)

n <- nrow(flights_clean)
prop <- round(mean(flights_clean$retard), 4)

cat("Proportion de vols en retard :", prop * 100, "%")

# carrier et origin sont des variables catégorielles : on les encode en indicatrices (one-hot) puisque xgboost n'accepte que des données numériques.
dummies <- caret::dummyVars(retard ~ ., data = flights_clean, fullRank = TRUE)

## 2. Séparation en échantillons train et test

In [ ]:
set.seed(123)
idx <- as.vector(caret::createDataPartition(factor(flights_clean$retard), p = 0.70, list = FALSE))

train_flights <- flights_clean[idx, ]
test_flights <- flights_clean[-idx, ]

cat("Proportion de retard - train :", round(mean(train_flights$retard), 4) * 100, "%\n")
cat("Proportion de retard - test  :", round(mean(test_flights$retard), 4) * 100, "%")

X_train <- predict(dummies, newdata = train_flights)
y_train <- train_flights$retard
X_test <- predict(dummies, newdata = test_flights)
y_test <- test_flights$retard

**Question** : Pourquoi s'assure-t-on d'avoir la même proportion de retard dans le train et le test ?

<details>
<summary>Voir la réponse</summary>

On utilise un tirage stratifié (`createDataPartition` sur `retard`) plutôt qu'un tirage purement aléatoire pour que train et test aient la même proportion de vols en retard (~24 %).

Sans cette précaution, un tirage aléatoire pourrait donner par hasard un test set avec une proportion de retards assez différente de celle du train, surtout que la classe "retard" est minoritaire. Les métriques (AUC, accuracy) seraient alors moins fiables et moins comparables d'une exécution à l'autre, car une partie de la variation viendrait du déséquilibre train/test plutôt que du modèle.

</details>

## 3. Premier modèle XGBoost (paramètres fixés)

Entraînement d'un premier modèle de classification binaire, avec un taux d'apprentissage de 0.1 et 100 arbres, en suivant l'AUC d'apprentissage.

In [ ]:
nb_coeurs <- parallel::detectCores()

boost_1 <- xgboost(data = X_train,
                   label = y_train,
                   nrounds = 100,
                   verbose = 0,
                   params = list(tree_method = "hist",
                                 objective = "binary:logistic",
                                 eval_metric = "auc",
                                 eta = 0.1,
                                 nthread = nb_coeurs - 1))

plot(boost_1$evaluation_log,
     xlab = "Nombre d'arbres", ylab = "AUC",
     main = "Evolution de l'AUC d'apprentissage",
     col = "red", type = "l")
grid(nx = NULL, ny = NULL, col = "lightgray")

**Question** : Comment évolue l'AUC d'apprentissage quand le nombre d'arbres augmente ? Cette AUC mesurée sur les données d'apprentissage permet-elle de conclure sur la qualité du modèle sur de nouvelles données ?

<details>
<summary>Voir la réponse</summary>

L'`evaluation_log` montre l'AUC d'apprentissage passer d'environ 0.91 (1 arbre) à environ 0.96 (100 arbres) : le modèle apprend progressivement.

Mais cette AUC est mesurée sur l'apprentissage : elle peut cacher du surapprentissage. On ne peut pas conclure sur de nouvelles données avec elle. C'est exactement ce qu'on va explorer à la section suivante, puis corriger par validation croisée.

</details>

## 4. Comprendre les hyperparamètres

Avant de laisser un algorithme régler le modèle, regardons **ce que fait chaque hyperparamètre**. On écrit une fonction qui entraîne un modèle et affiche l'**AUC sur le train ET sur le test** : comparer les deux est la clé pour repérer le surapprentissage.

In [ ]:
evaluer <- function(eta = 0.1, max_depth = 6, nrounds = 100) {
  m <- xgboost(data = X_train, label = y_train,
               objective = "binary:logistic",
               eta = eta, max_depth = max_depth, nrounds = nrounds,
               nthread = nb_coeurs - 1, verbose = 0)
  a_train <- auc(roc(y_train, predict(m, X_train), quiet = TRUE))
  a_test  <- auc(roc(y_test,  predict(m, X_test),  quiet = TRUE))
  cat("eta =", eta, "| max_depth =", max_depth, "| nrounds =", nrounds, "\n")
  cat("   AUC train =", round(a_train, 4), " | AUC test =", round(a_test, 4), "\n")
}

# Référence
evaluer()

### 4.1 Le taux d'apprentissage (`eta`)

`eta` règle le poids de chaque nouvel arbre dans l'ensemble.

**Question à la salle** : avec un `eta` très petit (0.01) et seulement 100 arbres, le modèle apprend-il mieux ou moins bien qu'avec `eta = 0.3` ?

In [ ]:
evaluer(eta = 0.01)
evaluer(eta = 0.3)

<details>
<summary>Voir la réponse</summary>

`eta` fixe la vitesse d'apprentissage. Avec `eta = 0.01` et seulement 100 arbres, chaque arbre contribue très peu : le modèle n'a pas le temps de converger et sous-apprend (AUC train et test plus basses). Avec `eta = 0.3`, il apprend plus vite et atteint de meilleures AUC en 100 arbres. Un `eta` plus petit n'est pas mauvais en soi : il faut simplement plus d'arbres pour compenser. Petit `eta` + beaucoup d'arbres donne souvent une meilleure généralisation, mais coûte plus cher en calcul.

</details>

### 4.2 La profondeur des arbres (`max_depth`)

`max_depth` contrôle la complexité de chaque arbre.

**Question à la salle** : avec des arbres très profonds (`max_depth = 12`), le modèle va-t-il forcément mieux généraliser ?

In [ ]:
evaluer(max_depth = 3)
evaluer(max_depth = 12)

<details>
<summary>Voir la réponse</summary>

Avec `max_depth = 12`, l'AUC train grimpe très haut mais l'AUC test stagne ou baisse : l'écart entre les deux se creuse. C'est du **surapprentissage** — les arbres profonds mémorisent le bruit du train au lieu de capturer des règles générales. Des arbres plus modestes (`max_depth = 3`) généralisent souvent mieux. La bonne profondeur se choisit, elle ne se maximise pas.

</details>

### 4.3 Le nombre d'arbres (`nrounds`)

**Question à la salle** : plus on ajoute d'arbres, mieux c'est ? Jusqu'où ?

In [ ]:
evaluer(nrounds = 10)
evaluer(nrounds = 500)

<details>
<summary>Voir la réponse</summary>

De 10 à 500 arbres, l'AUC train continue de monter (elle peut presque atteindre 1). L'AUC test progresse d'abord nettement, puis plafonne : **le gain est décroissant** et au-delà d'un certain point, ajouter des arbres n'améliore plus le test (et peut le dégrader). D'où l'intérêt d'un critère automatique pour s'arrêter au bon moment.

</details>

Régler ces trois leviers à la main est fastidieux. On laisse maintenant la **validation croisée** choisir, en commençant par le nombre d'arbres.

## 5. Sélection du nombre d'arbres par validation croisée

In [ ]:
X_mat <- xgb.DMatrix(X_train, label = y_train)

set.seed(123)
sel_xgb <- xgb.cv(data = X_mat,
                  tree_method = "hist",
                  params = list(objective = "binary:logistic",
                                eta = 0.1,
                                nthread = nb_coeurs - 1),
                  nrounds = 300,
                  metrics = "auc",
                  nfold = 10,
                  early_stopping_rounds = 10,
                  verbose = 0)

sel_xgb$evaluation_log |> head()
(ite_opt_demo <- sel_xgb$best_iteration)
sel_xgb$niter

**Question** : Que représente `ite_opt_demo` ? Pourquoi peut-il être inférieur à 300 ?

<details>
<summary>Voir la réponse</summary>

`ite_opt_demo` est le nombre d'arbres optimal (meilleure itération) sélectionné par validation croisée pour la profondeur par défaut : l'itération où l'AUC moyenne de test (sur les folds) est maximale.

Il peut être inférieur à 300 grâce à `early_stopping_rounds = 10` : la CV arrête l'entraînement dès que l'AUC de test ne s'améliore plus pendant 10 itérations consécutives, ce qui évite d'ajouter des arbres inutiles (et de surapprendre). C'est la version automatique de ce qu'on observait à la main en section 4.3.

</details>

### 5.1 Validation croisée sur le nombre d'arbres et la profondeur

In [ ]:
profondeurs <- c(2, 3, 4, 6, 8)

resultats_depth <- do.call(rbind, lapply(profondeurs, function(d) {
  cv_d <- xgb.cv(data = X_mat,
                 tree_method = "hist",
                 params = list(objective = "binary:logistic",
                               eta = 0.1,
                               max_depth = d,
                               nthread = nb_coeurs - 1),
                 nrounds = 300,
                 metrics = "auc",
                 nfold = 10,
                 early_stopping_rounds = 10,
                 verbose = 0)

  data.frame(max_depth = d,
             nrounds_opt = cv_d$best_iteration,
             auc_cv = cv_d$evaluation_log$test_auc_mean[cv_d$best_iteration])
}))

resultats_depth

meilleur <- resultats_depth[which.max(resultats_depth$auc_cv), ]
profondeur_opt <- meilleur$max_depth
ite_opt <- meilleur$nrounds_opt

cat("Profondeur optimale :", profondeur_opt,
    "- Nombre d'arbres optimal :", ite_opt,
    "- AUC CV :", round(meilleur$auc_cv, 4), "\n")

Pour faire la validation croisée aussi sur `gamma`, `lambda` et `alpha`, c'est le même principe : on crée une grille de toutes les combinaisons avec `expand.grid`. Ce n'est pas forcément utile ici (on a déjà une AUC de ~0.95 et le gain est généralement faible), et le temps de calcul augmente très vite.

## 6. Modèle final et performance sur le test

In [ ]:
boost_opt <- xgboost(data = X_train,
                     label = y_train,
                     tree_method = "hist",
                     objective = "binary:logistic",
                     eta = 0.1,
                     max_depth = profondeur_opt,
                     nrounds = ite_opt,
                     nthread = nb_coeurs - 1,
                     verbose = 0)

prev_opt <- predict(boost_opt, newdata = X_test)

roc_flights <- roc(y_test, prev_opt, quiet = TRUE)
cat("AUC test :", round(auc(roc_flights), 3), "\n")
cat("Accuracy test :", round(mean((prev_opt > 0.5) == y_test), 3), "\n")

plot(roc_flights, main = "Courbe ROC", col = "red")

**Question** : Comparez l'AUC de test avec l'AUC d'apprentissage du premier modèle. Que conclure sur le risque de surapprentissage du premier modèle par rapport au modèle final ?

<details>
<summary>Voir la réponse</summary>

L'AUC d'apprentissage du premier modèle (100 arbres fixés) atteignait ~0.96-0.965 sur le train lui-même, chiffre optimiste car mesuré sur les données ayant servi à ajuster le modèle.

L'AUC de test du modèle final (profondeur et nombre d'arbres choisis par CV) est ~0.95, très proche de l'AUC de CV (~0.948). Cette proximité test/CV, et l'écart avec l'AUC d'apprentissage brute du premier modèle, montrent que la validation croisée a choisi des hyperparamètres qui limitent le surapprentissage : le modèle final généralise bien, alors que le premier (100 arbres fixes, sans contrôle de profondeur ni early stopping) était plus à risque.

</details>

**Question** : L'accuracy utilise ici un seuil de 0.5 pour transformer la probabilité en décision. Ce seuil est-il forcément le plus adapté ?

<details>
<summary>Voir la réponse</summary>

Non, pas forcément. Le seuil de 0.5 n'a rien de spécial : il revient à classer "retard" dès que la probabilité dépasse 50 %, ce qui n'est optimal que si les deux types d'erreurs (faux positifs, faux négatifs) ont le même coût et que les classes sont équilibrées.

Ici, ~24 % des vols seulement sont en retard : les classes sont déséquilibrées, donc un seuil de 0.5 peut conduire à rarement prédire la classe minoritaire, même si le modèle discrimine bien (AUC élevée). Le seuil devrait dépendre du contexte métier (coût d'un faux négatif vs un faux positif) plutôt que d'être fixé par défaut à 0.5.

</details>

## 7. Comparaison avec le Gradient Boosting (`gbm`)

On compare la vitesse puis les performances des deux modèles avec les mêmes hyperparamètres.

In [ ]:
n_trees <- 300
depth <- 6
learning_rate <- 0.1

time_gbm <- system.time({
  model_gbm <- gbm(retard ~ ., data = train_flights,
                   distribution = "bernoulli",
                   n.trees = n_trees, interaction.depth = depth,
                   shrinkage = learning_rate, verbose = 0)
})

time_xgb <- system.time({
  model_xgb <- xgb.train(data = X_mat, tree_method = "hist",
                         objective = "binary:logistic",
                         nrounds = n_trees, max_depth = depth,
                         eta = learning_rate, nthread = 1, verbose = 0)
})

time <- data.frame(modele = c("gbm", "xgboost"),
                   temps_sec = c(time_gbm[["elapsed"]], time_xgb[["elapsed"]]))
kable(time, digits = 3, caption = "Temps d'entraînement (secondes)")

**Question** : Quel algorithme s'entraîne le plus vite à hyperparamètres égaux ? Cela correspond-il à un avantage classiquement mis en avant pour XGBoost ?

<details>
<summary>Voir la réponse</summary>

XGBoost s'entraîne nettement plus vite (quelques secondes contre ~19 s pour `gbm`), soit un facteur d'environ 4 à 7. Oui, c'est l'un des avantages classiques de XGBoost : implémentation optimisée et parallélisée (`tree_method = "hist"`, plusieurs cœurs via `nthread`), alors que `gbm` utilise un algorithme plus classique, séquentiel et mono-thread.

</details>

In [ ]:
pred_gbm <- predict(model_gbm, newdata = test_flights, n.trees = n_trees, type = "response")
pred_xgb <- predict(model_xgb, newdata = X_test)

roc_gbm <- roc(y_test, pred_gbm, quiet = TRUE)
roc_xgb <- roc(y_test, pred_xgb, quiet = TRUE)

perf <- data.frame(
  modele   = c("gbm", "xgboost"),
  AUC      = c(auc(roc_gbm), auc(roc_xgb)),
  accuracy = c(mean((pred_gbm > 0.5) == y_test),
               mean((pred_xgb > 0.5) == y_test))
)
kable(perf, digits = 3, caption = "Performance sur le jeu de test")

plot(roc_gbm, col = "blue", main = "Courbes ROC — gbm vs xgboost")
lines(roc_xgb, col = "red")
legend("bottomright", legend = c("gbm", "xgboost"), col = c("blue", "red"), lwd = 2)

**Question** : Comparez l'AUC et l'accuracy des deux modèles. Les différences sont-elles importantes ? XGBoost sera-t-il toujours plus performant que le Gradient Boosting classique ?

<details>
<summary>Voir la réponse</summary>

On obtient AUC ≈ 0.945 / accuracy ≈ 0.914 pour `gbm`, contre AUC ≈ 0.950 / accuracy ≈ 0.917 pour `xgboost`. Les écarts sont faibles (~0.005 d'AUC) et probablement peu significatifs : ils entrent dans la variabilité entre deux exécutions ou deux échantillonnages.

Non, on ne peut pas conclure que XGBoost sera toujours meilleur. Les deux modèles ont été entraînés avec exactement les mêmes hyperparamètres, sans réglage propre à chacun, sur un seul jeu de données. Les performances dépendent fortement des données, du réglage et de la régularisation : ailleurs, ou avec un `gbm` mieux réglé, les résultats pourraient s'inverser. L'avantage de XGBoost tient surtout à des aspects pratiques (vitesse, gestion native des valeurs manquantes, régularisation intégrée, flexibilité) plutôt qu'à une supériorité garantie en pouvoir prédictif brut.

</details>

## Conclusion

- **`eta`** (taux d'apprentissage) : trop petit → apprentissage lent (sous-apprentissage si peu d'arbres) ; plus grand → apprend plus vite.
- **`max_depth`** : trop grand → arbres complexes qui surapprennent (AUC train ↑↑, AUC test qui stagne ou baisse).
- **`nrounds`** : plus d'arbres aide, mais à rendement décroissant ; au-delà d'un point, le test ne progresse plus.
- **Validation croisée + arrêt anticipé** : choisit automatiquement un bon nombre d'arbres et une bonne profondeur, sans tâtonner.
- **XGBoost vs gbm** : à hyperparamètres égaux, performances proches ; l'avantage de XGBoost est surtout pratique (vitesse, régularisation, valeurs manquantes).